<!-- STEP X6 -->
# Optional extras (outside the timed hand-in run)

The extras below go beyond the brief. They are run separately from the main notebook, so the main
notebook stays within the five-minute limit. Three were chosen because they test the project's own
conclusions: **X6** (is the model only a fingerprint detector?), **X2** (does pseudo-labelling help with
far fewer labels?) and **X9** (how precise are the test-set numbers?).

In [1]:
# STANDIN A0
# Load the data, the split and the shared helpers (parts/00_setup.ipynb). Dropped from the hand-in
# notebook, where the setup cells come first.
%run 00_setup.ipynb

Python 3.13.7, scikit-learn 1.6.1, numpy 2.5.3, pandas 3.0.6
Working directory: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project
Data folder: /home/steven/Desktop/AI-for-Cybersecurity-Final-Mini-Project/UNSW-NB15
score() and score_proba() ready


Rows with an infinite or missing value: 0
Loaded 447,915 flows x 76 features


Raw rows:                      447,915
Exact duplicates removed:      141,742
Conflicting rows removed:        1,068
Clean rows (all are used):     305,105


Dropped 9 constant columns: Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, URG Flag Count, CWR Flag Count, ECE Flag Count, Fwd Bytes/Bulk Avg, Fwd Packet/Bulk Avg, Fwd Bulk Rate Avg
67 features: 65 continuous, 2 0/1 columns (Fwd PSH Flags, RST Flag Count)
TUNE_IDX: 36,612 training rows for settings searches
Setup complete: 22 shared names ready in 30 s


In [2]:
# STANDIN B2
# The three Part B models with the settings chosen on validation in B1 and B2
# (parts/10_baseline_models.ipynb). Dropped from the hand-in notebook, where the real B1 and B2 cells come first.
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

BOOST_SETTINGS = {"learning_rate": 0.1, "max_leaf_nodes": 31, "random_state": SEED}
def make_boost():
    return HistGradientBoostingClassifier(**BOOST_SETTINGS)

tree = DecisionTreeClassifier(max_depth=12, random_state=SEED).fit(X_train, y_train)
logreg = Pipeline([("scale", StandardScaler()),
                   ("model", LogisticRegression(C=10.0, max_iter=2000, random_state=SEED))]).fit(X_train, y_train)
boost = make_boost().fit(X_train, y_train)
MODELS = {"tree": tree, "logreg": logreg, "boosting": boost}

In [3]:
# STANDIN E1
# add_noise from step E1 (parts/40_robustness.ipynb). Dropped from the hand-in notebook.
def add_noise(X, level, seed=SEED):
    noisy = X.copy()
    if level == 0:
        return noisy
    rng = np.random.default_rng(seed)
    values = noisy[CONT_COLS].to_numpy(dtype=float, copy=True)
    values += rng.standard_normal(values.shape) * (level * TRAIN_STD[CONT_COLS].to_numpy())
    noisy[CONT_COLS] = np.maximum(values, 0.0)
    return noisy

<!-- STEP X2 -->
## X2: pseudo-labelling with far fewer labels

Part C found that pseudo-labelling did **not** help with 10% of the labels (18,306 flows), and gave as the
main reason that 10% of this dataset is already plenty: the lower line was only 0.003 macro-F1 below the
model with all labels. X2 tests that explanation: with **5% (9,153 flows) and 1% (1,830 flows)** of the
labels, the lower line should be weaker, leaving more room for the unlabelled flows to help.

Exactly the Part C method for each budget: a stratified share of the **training** set keeps its labels
(seed 42), the rest is unlabelled (its true labels are only used to count right guesses); the lower line
is gradient boosting on the labelled flows; pseudo-labelling runs 2 rounds with the cutoff chosen on
validation from 0.99, 0.95 and 0.90. The 10% results of Part C are repeated here for comparison. One label
draw per budget (X9 shows how precise single numbers are).

In [4]:
# STEP X2
def pseudo_label_from(X_lab, y_lab, X_unlab, y_hidden, cutoff, rounds=2):
    """Part C's pseudo-labelling for any label budget. Returns the final model and the share of right guesses."""
    X_now, y_now, pool = X_lab, y_lab, X_unlab
    added = right = 0
    for _ in range(rounds):
        model = make_boost().fit(X_now, y_now)
        p = model.predict_proba(pool)[:, 1]
        sure = (p >= cutoff) | (p <= 1 - cutoff)
        guess = pd.Series((p[sure] >= 0.5).astype(int), index=pool.index[sure])
        added += len(guess)
        right += int((guess == y_hidden.loc[guess.index]).sum())        # check only, never trained on
        X_now, y_now, pool = pd.concat([X_now, pool[sure]]), pd.concat([y_now, guess]), pool[~sure]
    return make_boost().fit(X_now, y_now), added, (right / added if added else np.nan)


rows, x2_models = [], {}            # x2_models: budget -> (lower line, pseudo-labelling), used by X9
for budget in [0.01, 0.05, 0.10]:
    X_lab, X_unlab, y_lab, y_hidden = train_test_split(X_train, y_train, train_size=budget,
                                                       stratify=type_train, random_state=SEED)
    lower = make_boost().fit(X_lab, y_lab)
    candidates = {}
    for cutoff in [0.99, 0.95, 0.90]:
        model, added, right = pseudo_label_from(X_lab, y_lab, X_unlab, y_hidden, cutoff)
        candidates[cutoff] = (model, added, right, score(model, X_val, y_val)["macro_f1"])
    # Cutoff with the best validation macro-F1 (strictest on a tie), as in C2.
    best = max(candidates, key=lambda c: (round(candidates[c][3], 4), c))
    pseudo, added, right, _ = candidates[best]
    x2_models[budget] = (lower, pseudo)
    for name, model in (("lower line", lower), (f"pseudo-labelling (cutoff {best})", pseudo)):
        for part, X, y in (("validation", X_val, y_val), ("test", X_test, y_test)):
            rows.append({"labels": f"{budget:.0%} ({len(X_lab):,})", "model": name, "set": part,
                         **score(model, X, y),
                         "guesses_added": added if name.startswith("pseudo") else 0,
                         "guesses_right": right if name.startswith("pseudo") else np.nan})
for part, X, y in (("validation", X_val, y_val), ("test", X_test, y_test)):
    rows.append({"labels": "100% (183,063)", "model": "upper line", "set": part, **score(boost, X, y),
                 "guesses_added": 0, "guesses_right": np.nan})

x2 = pd.DataFrame(rows)
x2.to_csv(TABLES / "X2_label_budgets.csv", index=False, float_format="%.4f")
x2.query("set == 'test'")[["labels", "model", "macro_f1", "recall", "pr_auc", "far", "guesses_added",
                            "guesses_right"]].round(4)

,labels,model,macro_f1,recall,pr_auc,far,guesses_added,guesses_right
1,"1% (1,830)",lower line,0.9603,0.9758,0.9550,0.0341,0,NaN
3,"1% (1,830)",pseudo-labelling (cutoff 0.9),0.9630,0.9881,0.9599,0.0358,179976,0.9732
5,"5% (9,153)",lower line,0.9666,0.9799,0.9746,0.0286,0,NaN
7,"5% (9,153)",pseudo-labelling (cutoff 0.9),0.9675,0.9891,0.9721,0.0311,170081,0.9814
9,"10% (18,306)",lower line,0.9688,0.9872,0.9755,0.0290,0,NaN
11,"10% (18,306)",pseudo-labelling (cutoff 0.95),0.9685,0.9894,0.9735,0.0301,150327,0.9907
13,"100% (183,063)",upper line,0.9718,0.9923,0.9814,0.0276,0,NaN


In [5]:
# STEP X2
# Did the unlabelled data help? Pseudo-labelling minus the lower line, per budget, on both sets.
diff = []
for (labels, part), g in x2[x2["model"] != "upper line"].groupby(["labels", "set"], sort=False):
    lower = g[g["model"] == "lower line"].iloc[0]
    pseudo = g[g["model"] != "lower line"].iloc[0]
    diff.append({"labels": labels, "set": part, **{m: pseudo[m] - lower[m]
                                                  for m in ["macro_f1", "recall", "pr_auc", "far"]}})
x2_diff = pd.DataFrame(diff)
x2_diff.to_csv(TABLES / "X2_did_it_help.csv", index=False, float_format="%.4f")
x2_diff.round(4)

,labels,set,macro_f1,recall,pr_auc,far
0,"1% (1,830)",validation,0.0028,0.0118,0.0033,0.0014
1,"1% (1,830)",test,0.0027,0.0123,0.0049,0.0017
2,"5% (9,153)",validation,0.0008,0.0087,-0.0004,0.0024
3,"5% (9,153)",test,0.0009,0.0092,-0.0025,0.0025
4,"10% (18,306)",validation,0.0001,0.0037,-0.0016,0.0013
5,"10% (18,306)",test,-0.0003,0.0022,-0.0020,0.0011


<!-- STEP X2 -->
### What fewer labels show

| Test set | Lower line macro-F1 | Pseudo-labelling macro-F1 | Gain | Share of the gap to all labels closed | Recall gain | FAR change |
|---|---:|---:|---:|---:|---:|---:|
| 1% labels (1,830 flows) | 0.960 | 0.963 | +0.003 | 23% | +0.012 | +0.002 |
| 5% labels (9,153 flows) | 0.967 | 0.968 | +0.001 | 17% | +0.009 | +0.003 |
| 10% labels (18,306 flows, Part C) | 0.969 | 0.969 | 0.000 | none | +0.002 | +0.001 |
| All labels (183,063 flows) | 0.972 | | | | | |

(Validation shows the same pattern: +0.003, +0.001 and 0.000 macro-F1.)

**Part C's explanation holds: the fewer the labels, the more the unlabelled data helps.** With 1% of the
labels, pseudo-labelling closes about a quarter of the gap to the model with all labels; with 10% it closes
none. The chosen cutoff also loosens as labels get scarce (0.90 at 1% and 5%, 0.95 at 10%), and the
guesses get less reliable (97.3% right at 1%, 99.1% at 10%).

**But the help stays small, and it is mostly recall.** Even with 1,830 labelled flows the lower line
already reaches a macro-F1 of 0.960: this dataset is easy to learn from a few labels, largely because of
the strong signals found in Part D. At 1%, pseudo-labelling catches 1.2% more of the attacks but also
raises a few more false alarms (FAR +0.002), the same trade-off as in Part C.

**Verdict.** "Did the unlabelled data help?" depends on how scarce labels are: at 10% no, at 1% a little
(+0.003 macro-F1, +0.012 recall). X9 shows that the 1% gain is larger than the noise of a single test set,
while the 10% difference is not. Each budget used one label draw, though, and the bootstrap does not cover
that, so the gains are best read as a trend rather than as exact values.

<!-- STEP X6 -->
## X6: the shortcut check, the black box without the attacker fingerprint

Parts D, E and G all found that the black box leans on the **TCP settings of the attacker machines**
(`Fwd Seg Size Min`, `FWD Init Win Bytes`, and on the victim side `Bwd Init Win Bytes`). Is it *only* a
fingerprint detector, or does it also learn the attacks' behaviour? We retrain gradient boosting
**without these three features**, with the same settings as B2 (not re-tuned), and compare it with the
original on four questions:

1. **How good is it on clean data** (validation and test)?
2. **How robust is it to measurement noise** (the E1 noise, levels 0.001 to 0.05)?
3. **Does it notice a new attack better** (the G1/G2 test: Fuzzers hidden from training)?
4. **What does it rely on instead** (its SHAP top 10, on the same 1,000 validation flows as D1)?

In [6]:
# STEP X6
TCP_FINGERPRINT = ["Fwd Seg Size Min", "FWD Init Win Bytes", "Bwd Init Win Bytes"]
NO_FP = [f for f in FEATURES if f not in TCP_FINGERPRINT]
boost_nofp = make_boost().fit(X_train[NO_FP], y_train)

# 1. Clean data.
x6_clean = pd.concat({
    part: pd.DataFrame({"with fingerprint (67 features)": score(boost, X, y),
                        "without fingerprint (64 features)": score(boost_nofp, X[NO_FP], y)}).T
    for part, X, y in (("validation", X_val, y_val), ("test", X_test, y_test))},
    names=["set", "model"])[["macro_f1", "recall", "pr_auc", "far"]]
x6_clean.to_csv(TABLES / "X6_clean.csv", float_format="%.4f")

# 2. Noise (same noise as E1, on the test set).
rows = []
for level in [0.0, 0.001, 0.005, 0.01, 0.02, 0.05]:
    X_noisy = add_noise(X_test, level)
    for name, model, cols in (("with fingerprint", boost, FEATURES), ("without fingerprint", boost_nofp, NO_FP)):
        rows.append({"level": level, "model": name, **score(model, X_noisy[cols], y_test)})
x6_noise = pd.DataFrame(rows)
x6_noise.to_csv(TABLES / "X6_noise.csv", index=False, float_format="%.4f")

print(f"Training without: {', '.join(TCP_FINGERPRINT)} ({len(NO_FP)} features left)")
print("\nRecall on the noisy test set:")
print(x6_noise.pivot(index="model", columns="level", values="recall").round(4).to_string())
x6_clean.round(4)

Training without: Fwd Seg Size Min, FWD Init Win Bytes, Bwd Init Win Bytes (64 features left)

Recall on the noisy test set:
level                 0.000   0.001   0.005   0.010   0.020   0.050
model                                                              
with fingerprint     0.9923  0.5128  0.4637  0.4604  0.4359  0.3035
without fingerprint  0.9910  0.8310  0.7892  0.7499  0.6409  0.4082


macro_f1  recall  pr_auc     far
set        model                                                              
validation with fingerprint (67 features)       0.9713  0.9924  0.9807  0.0282
           without fingerprint (64 features)    0.9711  0.9930  0.9806  0.0286
test       with fingerprint (67 features)       0.9718  0.9923  0.9814  0.0276
           without fingerprint (64 features)    0.9709  0.9910  0.9817  0.0281

In [7]:
# STEP X6
# 3. The hidden attack (G1/G2): Fuzzers removed from training, with and without the fingerprint.
HIDDEN = "Fuzzers"
keep = (type_train != HIDDEN).to_numpy()
is_hidden = (type_test == HIDDEN).to_numpy()
is_attack = (y_test == 1).to_numpy()

boost_hidden = make_boost().fit(X_train[keep], y_train[keep])
boost_nofp_hidden = make_boost().fit(X_train[keep][NO_FP], y_train[keep])
variants = [("with fingerprint", "all types", boost, FEATURES),
            ("with fingerprint", "Fuzzers hidden", boost_hidden, FEATURES),
            ("without fingerprint", "all types", boost_nofp, NO_FP),
            ("without fingerprint", "Fuzzers hidden", boost_nofp_hidden, NO_FP)]

rows = []
for name, trained_on, model, cols in variants:
    pred = model.predict(X_test[cols])
    rows.append({"model": name, "trained on": trained_on, "fuzzers_recall": pred[is_hidden].mean(),
                 "overall_recall": pred[is_attack].mean(), "far": float(pred[~is_attack].mean())})
x6_hidden = pd.DataFrame(rows)
x6_hidden.to_csv(TABLES / "X6_hidden.csv", index=False, float_format="%.4f")
x6_hidden.round(4)

,model,trained on,fuzzers_recall,overall_recall,far
0,with fingerprint,all types,0.9875,0.9923,0.0276
1,with fingerprint,Fuzzers hidden,0.6876,0.8880,0.0189
2,without fingerprint,all types,0.9864,0.9910,0.0281
3,without fingerprint,Fuzzers hidden,0.6870,0.8850,0.0189


In [8]:
# STEP X6
# 4. What the model without the fingerprint relies on: SHAP on the same 1,000 validation flows as D1.
import shap

GLOSSARY = pd.read_csv(TABLES / "T4_feature_glossary.csv", index_col="feature")
sample = X_val.sample(1000, random_state=SEED)
sv_nofp = shap.TreeExplainer(boost_nofp)(sample[NO_FP])
rank_nofp = pd.Series(np.abs(sv_nofp.values).mean(axis=0), index=NO_FP).sort_values(ascending=False)
x6_shap = pd.DataFrame({"share_of_shap_weight": (rank_nofp / rank_nofp.sum()).head(10)})
x6_shap = x6_shap.join(GLOSSARY[["meaning", "direction"]])
x6_shap.index.name = "feature"
x6_shap.to_csv(TABLES / "X6_shap_top10.csv", float_format="%.4f")
x6_shap.style.format({"share_of_shap_weight": "{:.1%}"})

,share_of_shap_weight,meaning,direction
feature,,,
Bwd Packets/s,46.1%,How fast the responder sent packets,backward
PSH Flag Count,16.0%,"Packets with the PSH flag (""deliver this data now"")",both
Bwd Header Length,6.2%,Total header bytes of the responder's packets,backward
Total Bwd packets,2.4%,Number of packets the responder (server/victim) sent back,backward
Packet Length Max,2.3%,"Largest packet payload in the conversation, either direction",both
Bwd Packet Length Max,1.7%,Largest packet payload the responder sent,backward
Fwd Packets/s,1.6%,How fast the initiator sent packets,forward
Fwd Packet Length Std,1.5%,How much the initiator's packet sizes vary,forward
FIN Flag Count,1.5%,"Packets with the FIN flag (""I am finished, close the connection"")",both


<!-- STEP X6 -->
### What the shortcut check shows

| Gradient boosting, test set | With fingerprint (67 features) | Without fingerprint (64 features) |
|---|---:|---:|
| Macro-F1, clean | 0.972 | 0.971 |
| Recall, clean | 0.992 | 0.991 |
| PR-AUC, clean | 0.981 | 0.982 |
| FAR, clean | 0.028 | 0.028 |
| Recall at noise level 0.001 | 0.513 | **0.831** |
| Recall at noise level 0.01 | 0.460 | **0.750** |
| Recall at noise level 0.05 | 0.304 | 0.408 |
| Recall on Fuzzers when Fuzzers are hidden from training (G2) | 0.688 | 0.687 |

**1. The fingerprint is not needed.** Without the three TCP-settings features the model is practically as
good on clean data: macro-F1 0.971 against 0.972, recall 0.991 against 0.992, validation the same. The
model used the fingerprint because it was the easiest signal, not because it was the only one: the
behaviour of the flows carries the same information.

**2. Without it, the model is much more robust to noise**, though still not robust: at noise level 0.001 it
catches 83% of the attacks instead of 51%, at 0.01 75% instead of 46%. Part of the fragility found in E3
came from the exact fingerprint values; the rest remains (exact zeros and narrow thresholds elsewhere).

**3. It does not recognise the new attack any better.** With Fuzzers hidden from training, recall on Fuzzers
is 0.687 with or without the fingerprint. The fingerprint was not what made the model miss fuzzing, and
removing it does not help it generalise to new kinds of attack; G3's labels are still needed.

**4. It now relies on what the victim does.** Its SHAP top 3 are `Bwd Packets/s` (how fast the other side
replies, 46% of the SHAP weight), `PSH Flag Count` (16%) and `Bwd Header Length` (the victim's header
bytes, 6%). These describe the conversation, mostly the victim's side of it, which the attacker cannot set
directly (E4: Fixed). And the cheap evasion of E4, copying the benign machines' TCP settings, no longer
applies at all, because the model does not look at those settings.

**Conclusion.** Dropping the attacker-fingerprint features costs almost nothing in accuracy, makes the
detector clearly more robust to measurement errors, and removes the cheapest way an attacker could evade it.
**For deployment we would train without them.** This also matters outside UNSW-NB15: on another network the
attackers would use other machines with other TCP settings, so a model that relies on behaviour has a much
better chance of carrying over than one that recognises a few lab machines.

<!-- STEP X9 -->
## X9: how precise are the test-set numbers?

Every score in this project comes from one test set of 61,021 flows. A different test set of the same size
would give slightly different numbers, so a small difference between two models may be luck. The
**bootstrap** estimates how much: draw 61,021 test flows **with replacement** (some flows twice, some not
at all), score every model on that resample, and repeat **200 times** (seed 42). The middle 95% of the 200
scores is the **95% confidence interval**.

The resamples are the same for every model (a *paired* bootstrap), so we can also put an interval on the
**difference** between two models. If that interval contains 0, the difference is within the noise of a
single test set and should not be called real. We check the differences the report relies on: gradient
boosting against the two glass boxes (B3), against the model without the fingerprint (X6), and
pseudo-labelling against its lower line with 10% labels (Part C) and 1% labels (X2).

In [9]:
# STEP X9
from sklearn.metrics import average_precision_score

probs = {
    "gradient boosting": boost.predict_proba(X_test)[:, 1],
    "decision tree": tree.predict_proba(X_test)[:, 1],
    "logistic regression": logreg.predict_proba(X_test)[:, 1],
    "boosting without fingerprint (X6)": boost_nofp.predict_proba(X_test[NO_FP])[:, 1],
    "lower line, 10% labels (C1)": x2_models[0.10][0].predict_proba(X_test)[:, 1],
    "pseudo-labelling, 10% labels (C2)": x2_models[0.10][1].predict_proba(X_test)[:, 1],
    "lower line, 1% labels (X2)": x2_models[0.01][0].predict_proba(X_test)[:, 1],
    "pseudo-labelling, 1% labels (X2)": x2_models[0.01][1].predict_proba(X_test)[:, 1],
}
y_arr = y_test.to_numpy()


def fast_scores(y, p):
    """Macro-F1, recall, PR-AUC and FAR from labels and probabilities (threshold 0.5)."""
    pred = p >= 0.5
    tp = np.sum(pred & (y == 1)); fp = np.sum(pred & (y == 0))
    fn = np.sum(~pred & (y == 1)); tn = np.sum(~pred & (y == 0))
    f1_attack = 2 * tp / (2 * tp + fp + fn)
    f1_benign = 2 * tn / (2 * tn + fn + fp)
    return {"macro_f1": (f1_attack + f1_benign) / 2, "recall": tp / (tp + fn),
            "pr_auc": average_precision_score(y, p), "far": fp / (fp + tn)}


# Check: on the full test set the fast scores equal score_proba() from the setup.
for name, p in probs.items():
    ref = score_proba(y_test, p)
    assert all(np.isclose(fast_scores(y_arr, p)[m], ref[m]) for m in ["macro_f1", "recall", "pr_auc", "far"]), name

rng = np.random.default_rng(SEED)
boot = []
for b in range(200):
    idx = rng.integers(0, len(y_arr), len(y_arr))
    for name, p in probs.items():
        boot.append({"resample": b, "model": name, **fast_scores(y_arr[idx], p[idx])})
boot = pd.DataFrame(boot)

point = pd.DataFrame({name: fast_scores(y_arr, p) for name, p in probs.items()}).T
rows = []
for name in probs:
    for m in ["macro_f1", "recall", "pr_auc", "far"]:
        v = boot.loc[boot["model"] == name, m]
        rows.append({"model": name, "metric": m, "test_score": point.loc[name, m],
                     "ci_low": v.quantile(0.025), "ci_high": v.quantile(0.975)})
x9_ci = pd.DataFrame(rows)
x9_ci["ci_width"] = x9_ci["ci_high"] - x9_ci["ci_low"]
x9_ci.to_csv(TABLES / "X9_confidence_intervals.csv", index=False, float_format="%.4f")
x9_ci.pivot(index="model", columns="metric", values="ci_width").round(4)

metric,far,macro_f1,pr_auc,recall
model,,,,
boosting without fingerprint (X6),0.0029,0.0033,0.0038,0.0028
decision tree,0.0029,0.0030,0.0064,0.0034
gradient boosting,0.0029,0.0032,0.0042,0.0025
logistic regression,0.0034,0.0033,0.0072,0.0017
"lower line, 1% labels (X2)",0.0032,0.0037,0.0080,0.0039
"lower line, 10% labels (C1)",0.0031,0.0032,0.0046,0.0032
"pseudo-labelling, 1% labels (X2)",0.0035,0.0032,0.0071,0.0031
"pseudo-labelling, 10% labels (C2)",0.0030,0.0030,0.0051,0.0032


In [10]:
# STEP X9
# Paired differences: model A minus model B on the same resamples, with a 95% interval.
pairs = [("gradient boosting", "decision tree"), ("gradient boosting", "logistic regression"),
         ("gradient boosting", "boosting without fingerprint (X6)"),
         ("pseudo-labelling, 10% labels (C2)", "lower line, 10% labels (C1)"),
         ("pseudo-labelling, 1% labels (X2)", "lower line, 1% labels (X2)")]
wide = boot.pivot_table(index="resample", columns="model", values=["macro_f1", "recall", "pr_auc", "far"])
rows = []
for a, b in pairs:
    for m in ["macro_f1", "recall", "pr_auc", "far"]:
        d = wide[(m, a)] - wide[(m, b)]
        low, high = d.quantile(0.025), d.quantile(0.975)
        rows.append({"comparison": f"{a} minus {b}", "metric": m,
                     "difference": point.loc[a, m] - point.loc[b, m], "ci_low": low, "ci_high": high,
                     "real?": "yes" if (low > 0 or high < 0) else "no, within the noise"})
x9_diff = pd.DataFrame(rows)
x9_diff.to_csv(TABLES / "X9_differences.csv", index=False, float_format="%.4f")
x9_diff[x9_diff["metric"].isin(["macro_f1", "recall"])].round(4)

,comparison,metric,difference,ci_low,ci_high,real?
0,gradient boosting minus decision tree,macro_f1,0.0028,0.0020,0.0035,yes
1,gradient boosting minus decision tree,recall,0.0074,0.0057,0.0090,yes
4,gradient boosting minus logistic regression,macro_f1,0.0109,0.0099,0.0121,yes
5,gradient boosting minus logistic regression,recall,-0.0039,-0.0055,-0.0022,yes
8,gradient boosting minus boosting without finge...,macro_f1,0.0009,0.0004,0.0014,yes
9,gradient boosting minus boosting without finge...,recall,0.0012,0.0002,0.0022,yes
12,"pseudo-labelling, 10% labels (C2) minus lower ...",macro_f1,-0.0003,-0.0009,0.0004,"no, within the noise"
13,"pseudo-labelling, 10% labels (C2) minus lower ...",recall,0.0022,0.0007,0.0037,yes
16,"pseudo-labelling, 1% labels (X2) minus lower l...",macro_f1,0.0027,0.0019,0.0036,yes
17,"pseudo-labelling, 1% labels (X2) minus lower l...",recall,0.0123,0.0107,0.0143,yes


<!-- STEP X9 -->
### What the confidence intervals show

**A single test score is precise to about ±0.002.** With 61,021 test flows, the 95% intervals are about
0.003 wide for macro-F1 (gradient boosting: 0.970 to 0.973), 0.002 to 0.004 for recall and 0.003 for the
false alarm rate. Differences smaller than about 0.002 between two models should therefore not be trusted
without a paired comparison.

| Paired comparison (test set) | Macro-F1 difference (95% interval) | Real? |
|---|---|---|
| Gradient boosting minus decision tree | +0.0028 (+0.0020 to +0.0035) | yes |
| Gradient boosting minus logistic regression | +0.0109 (+0.0099 to +0.0121) | yes |
| With fingerprint minus without (X6) | +0.0009 (+0.0004 to +0.0014) | yes, but tiny |
| Pseudo-labelling minus lower line, 10% labels (Part C) | −0.0003 (−0.0009 to +0.0004) | **no, within the noise** |
| Pseudo-labelling minus lower line, 1% labels (X2) | +0.0027 (+0.0019 to +0.0036) | yes |

**What this settles for the report.**
- **Gradient boosting really is the best model**, but by little: 0.003 macro-F1 and 0.007 recall over the
  decision tree are real differences, while their false alarm rates cannot be told apart (difference
  −0.0003, interval −0.0007 to +0.0002).
- **Part C's "no" is confirmed.** With 10% of the labels, the macro-F1 difference between pseudo-labelling
  and the lower line is within the noise. What *is* real is the trade: pseudo-labelling catches slightly
  more attacks (recall +0.0022) and raises slightly more false alarms (FAR +0.0011), with a slightly worse
  ranking (PR-AUC −0.0020). With 1% of the labels the gain is real (+0.0027 macro-F1, +0.012 recall).
- **Dropping the fingerprint (X6) has a real but negligible cost** of 0.0009 macro-F1 and 0.0012 recall; its
  PR-AUC cannot be told apart. That supports training without it.

**What the bootstrap does not cover.** It only measures how much the scores would move with another test
set drawn from the *same* data. It says nothing about other training seeds or label draws (X2 used one
draw per budget), about small attack types (Worms has only 47 test flows, so its recall interval is far
wider than these), or about traffic from another network or another year. The intervals are narrow
because the test set is large; the bigger uncertainty lies in whether this 2015 lab data, with its few
attacker machines, resembles real traffic at all (Part D, X6).